# Manual oscillator initialization with `interactive_oscillator_guess`

This notebook shows how to use `empylib.nklib.interactive_oscillator_guess` to choose better starting parameters before calling `fit_to_oscillator`.

The helper is designed for Jupyter notebooks. It plots measured data on its original `y_data.index`, while the oscillator/model curves are evaluated on the `wavelength` grid you pass in. It returns a small controller object with a displayable widget and a current oscillator dictionary:

- `controller.widget`: the interactive sliders and plot output
- `controller.model`: the current oscillator dictionary
- `controller.get_model()`: explicit method returning the same dictionary
- `controller.close()`: closes the widget objects when you are done

If the widget does not display, make sure `ipywidgets` is installed and enabled in your notebook environment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from empylib import nklib as nk

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

: 

## 1) Create example refractive-index data

In a real workflow, `n_data` and `k_data` usually come from a `.nk` file, a database, or measurements. Here we create synthetic data from a known Lorentz oscillator so the notebook can run anywhere. The DataFrame index is the measured wavelength grid that will be plotted as-is.

In [ ]:
lam = np.linspace(0.45, 1.4, 160)  # wavelength in um

true_oscillator = {
    "lorentz_1": {"type": "lorentz", "epsinf": 1.0, "wp": 4.0, "wn": 2.35, "gamma": 0.18},
}

nk_true = nk.multi_oscillator(lam, true_oscillator)
n_data = nk_true.real
k_data = nk_true.imag
y_data = pd.DataFrame({"n": n_data, "k": k_data}, index=lam)
y_data.index.name = "wavelength"

fig, ax = plt.subplots()
ax.plot(lam, n_data, label="n data")
ax.plot(lam, k_data, label="k data")
ax.set_xlabel("wavelength (um)")
ax.set_ylabel("refractive index")
ax.legend();

## 2) Start from a rough oscillator guess

The initial guess uses the same dictionary structure required by `multi_oscillator` and `fit_to_oscillator`. The bounds below are optional; any missing parameter uses the same default bounds as `fit_to_oscillator`. You can pass a narrower `wavelength` grid to inspect a fit range without resampling the measured data.

In [ ]:
rough_guess = {
    "lorentz_1": {"type": "lorentz", "epsinf": 2.0, "wp": 1.2, "wn": 4.5, "gamma": 0.6},
}

bounds = {
    "lorentz_1": {
        "epsinf": (0.0, 4.0),
        "wp": (0.1, 8.0),
        "wn": (0.2, 7.0),
        "gamma": (0.01, 1.5),
    }
}

## 3) Tune parameters manually

Run the next cell, move the sliders until the dashed model curves are close to the solid data curves, then continue. The final selected values are read from `controller.model`.

In [ ]:
controller = nk.interactive_oscillator_guess(
    lam,
    y_data,
    rough_guess,
    bounds=bounds,
    figure_kwargs={"figsize": (8, 6)},
)

controller.widget

Read the current oscillator dictionary after adjusting the sliders. This dictionary can be passed directly to `fit_to_oscillator`.

In [ ]:
manual_guess = controller.model
manual_guess

## 4) Use the manual guess in `fit_to_oscillator`

The manual step gives the optimizer a much better starting point, which is especially helpful for oscillator models with several local minima.

In [ ]:
fitted, result = nk.fit_to_oscillator(
    lam,
    y_data,
    manual_guess,
    bounds=bounds,
)

print("success:", result.success)
print("cost:", result.cost)
fitted.model

In [ ]:
nk_fit = nk.multi_oscillator(lam, fitted.model)

fig, ax = plt.subplots()
ax.plot(lam, n_data, "C0-", label="n data")
ax.plot(lam, k_data, "C1-", label="k data")
ax.plot(lam, nk_fit.real, "C0--", label="n fit")
ax.plot(lam, nk_fit.imag, "C1--", label="k fit")
ax.set_xlabel("wavelength (um)")
ax.set_ylabel("refractive index")
ax.legend();

## 5) List input with automatic model names

`interactive_oscillator_guess` can also receive a list of typed model dictionaries. The helper converts them into a normal oscillator dictionary with deterministic names such as `drude_1`, `lorentz_1`, and `lorentz_2`.

In [ ]:
model_list_guess = [
    {"type": "lorentz", "epsinf": 2.0, "wp": 1.2, "wn": 4.5, "gamma": 0.6},
]

list_controller = nk.interactive_oscillator_guess(
    lam,
    y_data,
    model_list_guess,
    bounds={
        "lorentz_1": {"epsinf": (0.0, 4.0), "wp": (0.1, 8.0), "wn": (0.2, 7.0), "gamma": (0.01, 1.5)},
    },
)

list_controller.widget

## 6) Custom `y_eval` data

When `y_eval` is provided, the interactive helper plots the custom model outputs against `y_data`. In this first version, sliders still control only oscillator parameters; extra `y_eval` parameters are passed through `args`.

In [ ]:
def scaled_nk_outputs(lam_um, nk_values, scale):
    return [scale * nk_values.real, scale * nk_values.imag]

scale = 1.15
custom_data = scaled_nk_outputs(lam, nk_true, scale)
custom_y_data = pd.DataFrame({"scaled_n": custom_data[0], "scaled_k": custom_data[1]}, index=lam)
custom_y_data.index.name = "wavelength"

custom_controller = nk.interactive_oscillator_guess(
    lam,
    custom_y_data,
    rough_guess,
    y_eval=scaled_nk_outputs,
    args=(scale,),
    bounds=bounds,
    figure_kwargs={"figsize": (8, 6)},
)

custom_controller.widget

## 7) Cleanup

Closing widgets is optional in normal notebook use, but it is helpful when experimenting repeatedly in the same kernel.

In [ ]:
# controller.close()
# list_controller.close()
# custom_controller.close()